# 5_rag_tr — Türkiye Mevzuatı RAG Katmanı

Agent'ın kararlarını, resmî mevzuattaki kavram ve süreçlerle **gerekçelendirmesi** için. Karar eşikleri RAG'den gelmiyor; eşikler `agent_esikleri.json`'dan geliyor. Bunun nedeni, mevzuattaki eşiklerin MW değil frekans cinsinden olması.

- **A.** Kurulum ve belge kontrolü
- **B.** PDF'den metin
- **C.** Madde bazlı parçalama
- **D.** Embedding ve ChromaDB (tekrar çalıştırmaya dayanıklı)
- **E.** Getirme ve duruma duyarlı sorgu
- **F.** Getirme kalite testi
- **G.** Agent için paylaşılan modül (`notepads/tr_rag_retrieval.py`)

**PJM projesinden farklar:**

| Konu | PJM | Bu notebook |
|---|---|---|
| Parçalama | Sabit 1.000 karakter | Madde bazlı; madde numarası metaveride |
| Görev türü | Belirtilmemiş | Parça: `RETRIEVAL_DOCUMENT`, sorgu: `RETRIEVAL_QUERY` |
| Vektör boyutu | 3.072 (varsayılan) | 768, L2 normalize |
| Tekrar çalıştırma | Her seferinde yeniden ekleme | Yalnızca yeni veya değişen parçalar gömülüyor |

## A. Belgelerin Bulunması, Tanınması ve Kontrolü

Dosyalar **adlarına göre değil, içeriklerine göre** tanınıyor. Bu şu durumları çözüyor:
- İndirilen dosyaların adları kaynağa göre değişiyor (`7.5.19722.pdf`, `GeneratePdf.pdf`, EPDK'nın `_PortalAdmin_...docx` dosyaları).
- Türkçe karakterli dosya adları bilgisayarlar arasında farklı Unicode biçimlerinde saklanabiliyor.
- Yanlış belge indirilirse, örneğin yürürlükten kalkmış bir yönetmelik, bu fark edilebiliyor.

**Bölümler:**
- **A1:** Kurulum.
- **A2:** Okuyucular (PDF ve DOCX) ve tanıma.
- **A3:** `rag_docs/` ve `docs/` klasörlerinin taranması; eksik zorunlu belgelerin, adresi doğrulanmış olanlarının indirilmesi.
- **A4:** Belge bazında kimlik ve güncellik raporu. Hiç belge yoksa notebook durur.

In [1]:
import os
import re
import sys
import json
import time
import hashlib
import zipfile
import unicodedata
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from pypdf import PdfReader
import chromadb
from google import genai
from google.genai import types
from langsmith import traceable

KOK = r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus"
load_dotenv(os.path.join(KOK, "notepads", ".env"))
os.environ["LANGSMITH_PROJECT"] = "tr-grid-agent"      # PJM projesinden ayri LangSmith projesi
RAG_DIR = os.path.join(KOK, "rag_docs")
DOCS_DIR = os.path.join(KOK, "docs")
os.makedirs(RAG_DIR, exist_ok=True)

TURLER = {  # tur: (belge adi, zorunlu mu)
    "sebeke": ("Elektrik Şebeke Yönetmeliği", True),
    "dengeleme": ("Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği", True),
    "yan_hizmetler": ("Elektrik Piyasası Yan Hizmetler Yönetmeliği", True),
    "yan_hizmetler_degisiklik": ("Elektrik Piyasası Yan Hizmetler Yönetmeliğinde Değişiklik (RG 09.04.2026, 33219)", False),
    "veri_listesi": ("EPDK 10711 sayılı Kurul Kararı eki: Şeffaflık Platformu veri listesi", True),
}
KAYNAK_URL = {   # arastirmada dogrulanmis adresler; yalnizca belge bulunamazsa kullanilir
    "sebeke": "https://www.mevzuat.gov.tr/MevzuatMetin/yonetmelik/7.5.19722.pdf",
    "dengeleme": "https://www.mevzuat.gov.tr/File/GeneratePdf?mevzuatNo=12985&mevzuatTur=KurumVeKurulusYonetmeligi&mevzuatTertip=5",
    "veri_listesi": "https://www.epias.com.tr/wp-content/uploads/2022/01/S%CC%A7effaflik-Platformunda-Yayimlanacak-Veri-Listesi.pdf",
}
EMBED_MODEL = "gemini-embedding-001"
BOYUT = 768

### A2. Okuyucular ve tanıma

- **PDF:** `pypdf`.
- **DOCX:** Ek paket gerektirmeden, Word dosyasının içindeki XML'den paragraf paragraf okunuyor.

**Madde başlıkları:** "MADDE 12 –" biçiminin yanında ek maddeler ("MADDE 67/A –", "MADDE 7/A –") ve dipnot işaretli biçim ("MADDE 11 (1) –") de tanınıyor. İlk sürüm ek maddeleri tanımıyordu; bu yüzden ek maddeler bir önceki maddenin parçasında kalıyordu (`6_agent_tr` çıktısında görüldü).

**Tanıma:** Metnin ilk 6.000 karakterinde en önce geçen belge başlığı belirleyici. Değişiklik yönetmeliğinin başlığı ("...Yan Hizmetler Yönetmeliğinde Değişiklik Yapılmasına Dair...") ana yönetmeliğin başlığını da içerdiği için önce o kontrol ediliyor.

**Güncellik:** Mevzuat metinlerindeki "(Değişik:RG-17/12/2024-32755)" gibi notlardan, belgedeki **en son değişiklik tarihi** çıkarılıyor. Bu, indirilen sürümün hangi değişikliğe kadar güncel olduğunu gösteriyor.

In [2]:
# Ek maddeler (MADDE 67/A, 7/A, 32/B) ve dipnotlu bicim (MADDE 11 (1) –) da taniniyor
MADDE_RE = re.compile(r"(?m)^[ \t]*((?:GEÇİCİ|EK)[ \t]+MADDE|MADDE)[ \t]+(\d+(?:[ \t]*/[ \t]*[A-ZÇĞİÖŞÜ])?)(?:[ \t]*\(\d+\))?[ \t]*[–—-]")
RG_RE = re.compile(r"R\s*G\s*[-–]\s*(\d{1,2})\s*/\s*(\d{1,2})\s*/\s*(\d{4})")   # satir sonuyla bolunen notlar icin bosluga toleransli

def tr_kucuk(s):
    # Turkce kucuk harf ve bosluk normalizasyonu (PDF'lerde ifadeler satir sonuyla bolunebiliyor)
    s = unicodedata.normalize("NFC", s)
    return re.sub(r"\s+", " ", s.replace("İ", "i").replace("I", "ı").lower())

def temizle(metin):
    metin = unicodedata.normalize("NFC", metin)
    metin = re.sub(r"[ \t]+", " ", metin)
    return re.sub(r"\n{3,}", "\n\n", metin)

def pdf_metin(yol):
    return temizle("\n".join((s.extract_text() or "") for s in PdfReader(yol).pages))

def docx_metin(yol):
    w = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"
    with zipfile.ZipFile(yol) as z:
        kok_xml = ET.fromstring(z.read("word/document.xml"))
    paragraflar = []
    for p in kok_xml.iter(w + "p"):
        parca = []
        for el in p.iter():
            if el.tag == w + "t" and el.text:
                parca.append(el.text)
            elif el.tag in (w + "tab",):
                parca.append(" ")
            elif el.tag == w + "br":
                parca.append("\n")
        paragraflar.append("".join(parca))
    return temizle("\n".join(paragraflar))

def belge_metin(yol):
    return pdf_metin(yol) if yol.lower().endswith(".pdf") else docx_metin(yol)

BASLIKLAR = {"sebeke": "elektrik şebeke yönetmeliği",
             "dengeleme": "dengeleme ve uzlaştırma yönetmeliği",
             "yan_hizmetler": "yan hizmetler yönetmeliği",
             "veri_listesi": "yayımlanacak veri listesi"}

def tani(metin):
    bas = tr_kucuk(metin[:6000])
    if "yan hizmetler yönetmeliğinde değişiklik" in bas:
        return "yan_hizmetler_degisiklik"
    konum = {t: bas.find(b) for t, b in BASLIKLAR.items() if bas.find(b) >= 0}
    if konum:
        return min(konum, key=konum.get)
    tum = tr_kucuk(metin)
    return "veri_listesi" if ("s+2" in tum and "g-1" in tum) else None

def son_rg(metin):
    tarihler = [pd.Timestamp(int(y), int(a), int(g)) for g, a, y in RG_RE.findall(metin)
                if 1 <= int(a) <= 12 and 1 <= int(g) <= 31]
    return max(tarihler) if tarihler else None

def tara():
    bulunan = []
    for klasor in [RAG_DIR, DOCS_DIR]:
        if not os.path.isdir(klasor):
            continue
        for ad in sorted(os.listdir(klasor)):
            yol = os.path.join(klasor, ad)
            if not ad.lower().endswith((".pdf", ".docx")) or ad.startswith("~$"):
                continue
            try:
                m = belge_metin(yol)
            except Exception as e:
                bulunan.append({"tur": None, "yol": yol, "hata": str(e)[:100]})
                continue
            bulunan.append({"tur": tani(m), "yol": yol, "metin": m, "karakter": len(m),
                            "madde": len(MADDE_RE.findall(m)), "son_rg": son_rg(m), "hata": ""})
    return bulunan
print("Okuyucular ve tanima hazir.")

Okuyucular ve tanima hazir.


### A3. Tarama ve eksik belgelerin indirilmesi

Zorunlu bir belge iki klasörde de bulunamazsa, adresi doğrulanmış olanlar `rag_docs/` klasörüne indirilmeye çalışılıyor. İndirilen dosyanın ilk baytlarının `%PDF` olup olmadığı da kontrol ediliyor. Yan Hizmetler Yönetmeliği'nin doğrudan PDF adresi doğrulanamadığı için elle indirilmesi gerekiyor. Kaynak: EPDK'nın yönetmelik sayfası (`https://www.epdk.gov.tr/Detay/Icerik/3-6723/elektrik-piyasasi-yan-`) veya TEİAŞ'ın yönetmelikler sayfası.

In [3]:
bulunan = tara()
turler_var = {b["tur"] for b in bulunan if b.get("tur")}
for tur, url in KAYNAK_URL.items():
    if tur in turler_var:
        continue
    yol = os.path.join(RAG_DIR, f"tr_{tur}.pdf")
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=90)
        if r.status_code == 200 and r.content[:4] == b"%PDF":
            with open(yol, "wb") as fh:
                fh.write(r.content)
            print(f"Indirildi: {os.path.basename(yol)} ({len(r.content) / 1e6:.2f} MB)")
        else:
            print(f"INDIRILEMEDI: {tur} | HTTP {r.status_code} -> elle indirin: {url}")
    except requests.RequestException as e:
        print(f"INDIRILEMEDI: {tur} | {str(e)[:100]} -> elle indirin: {url}")
bulunan = tara()
print("Taranan belge dosyasi:", len(bulunan))

Taranan belge dosyasi: 7


### A4. Belge bazında kimlik ve güncellik raporu

Her dosya için tanınan tür, karakter sayısı, MADDE sayısı ve en son değişiklik tarihi yazdırılıyor. Aynı türden birden fazla dosya varsa, en son değişiklik tarihi en yeni olan, o da eşitse metni en uzun olan kullanılıyor.

**Özel kontroller:**
1. **Yan Hizmetler, yürürlükten kalkmış sürüm mü?** Eski (mülga) yönetmelik "Elektrik Piyasası Şebeke Yönetmeliği"ne atıf yapıyor. Güncel yönetmelik "Elektrik Şebeke Yönetmeliği"ne atıf yapıyor ve primer frekans kontrol rezervini "±200 mHz" ile tanımlıyor.
2. **Yan Hizmetler, 9 Nisan 2026 değişikliğini içeriyor mu?** Bu değişiklik yeni bir geçici madde (GEÇİCİ MADDE 12) ekledi. Bu madde yan hizmet performans testlerinin 1/9/2026'dan itibaren TEİAŞ tarafından yapılmasını düzenliyor. Belge bu değişikliği içermiyorsa, değişiklik yönetmeliğinin ayrıca indirilmesi öneriliyor.
3. **MADDE sayısı 0 olan yönetmelik:** Metin okunamamış olabilir (taranmış görüntü).

In [4]:
rapor = []
for b in bulunan:
    if b.get("hata"):
        rapor.append({"dosya": os.path.basename(b["yol"]), "klasor": os.path.basename(os.path.dirname(b["yol"])),
                      "tur": "OKUNAMADI", "karakter": 0, "madde": 0, "son_degisiklik": "", "not": b["hata"]})
        continue
    rapor.append({"dosya": os.path.basename(b["yol"]), "klasor": os.path.basename(os.path.dirname(b["yol"])),
                  "tur": b["tur"] or "TANINMADI", "karakter": b["karakter"], "madde": b["madde"],
                  "son_degisiklik": b["son_rg"].date() if b["son_rg"] is not None else "", "not": ""})
print(pd.DataFrame(rapor).to_string(index=False))

secilen = {}
for b in bulunan:
    if not b.get("tur"):
        continue
    onceki = secilen.get(b["tur"])
    anahtar = (b["son_rg"] or pd.Timestamp(1900, 1, 1), b["karakter"])
    if onceki is None or anahtar > (onceki["son_rg"] or pd.Timestamp(1900, 1, 1), onceki["karakter"]):
        secilen[b["tur"]] = b

print("\nKullanilacak belgeler:")
for tur, (ad, zorunlu) in TURLER.items():
    b = secilen.get(tur)
    durum = f"VAR  {os.path.basename(b['yol'])}" if b else ("YOK  (zorunlu)" if zorunlu else "YOK  (istege bagli)")
    print(f"  {tur:26s} {durum}")

uyarilar = []
yh = secilen.get("yan_hizmetler")
if yh:
    t = tr_kucuk(yh["metin"])
    if "elektrik piyasası şebeke yönetmeliği" in t and "200 mhz" not in t:
        uyarilar.append("Yan Hizmetler: MULGA (eski) surum olabilir. 26/11/2017 tarihli ve 30252 sayili RG'deki guncel yonetmeligi indirin.")
    if "200 mhz" not in t:
        uyarilar.append("Yan Hizmetler: '±200 mHz' tanimi bulunamadi; belge icerigini kontrol edin.")
    if "geçici madde 12" not in t and "yan_hizmetler_degisiklik" not in secilen:
        uyarilar.append("Yan Hizmetler: 09.04.2026 (RG 33219) degisikligi (GECICI MADDE 12) belgede yok ve degisiklik dosyasi bulunamadi. "
                        "EPDK sayfasindaki 'Degisiklik 09.04.2026 33219' dosyasini da docs klasorune indirin.")
for tur, b in secilen.items():
    if tur != "veri_listesi" and b["madde"] == 0:
        uyarilar.append(f"{tur}: MADDE basligi bulunamadi; metin okunamamis olabilir (taranmis PDF).")
for b in bulunan:
    if b.get("tur") is None and not b.get("hata"):
        uyarilar.append(f"Taninmayan dosya (kullanilmayacak): {os.path.basename(b['yol'])}")
print("\nUyarilar:" if uyarilar else "\nUyari yok.")
for u in uyarilar:
    print("  -", u)

eksik_zorunlu = [t for t, (_, z) in TURLER.items() if z and t not in secilen]
if not secilen:
    raise FileNotFoundError(f"Hic belge bulunamadi ({RAG_DIR} ve {DOCS_DIR}). A3'un ciktisina gore indirip tekrar calistirin.")
if eksik_zorunlu:
    print("\nEksik zorunlu belge(ler):", eksik_zorunlu, "- mevcut olanlarla devam ediliyor.")
metinler = {tur: b["metin"] for tur, b in secilen.items()}

                                                     dosya   klasor           tur  karakter  madde son_degisiklik not
                               tr_dengeleme_uzlastirma.pdf rag_docs     dengeleme    413884    229     2025-12-29    
                                 tr_sebeke_yonetmeligi.pdf rag_docs        sebeke    458620    126     2025-12-29    
                             tr_seffaflik_veri_listesi.pdf rag_docs  veri_listesi     25357      0                   
                                             7.5.19722.pdf     docs        sebeke    458620    126     2025-12-29    
                                           GeneratePdf.pdf     docs     dengeleme    413884    229     2025-12-29    
     Şeffaflik-Platformunda-Yayimlanacak-Veri-Listesi.pdf     docs  veri_listesi     25357      0                   
_PortalAdmin_Uploads_Content_FastAccess_389cc81749363.docx     docs yan_hizmetler    188682     90     2025-12-29    

Kullanilacak belgeler:
  sebeke                     VAR

## B. Metin İstatistikleri

Her belge için karakter sayısı ve MADDE başlığı sayısı, ayrıca ilk belgenin başından bir örnek.

In [5]:
for tur, m in metinler.items():
    print(f"{tur}: {len(m):,} karakter | MADDE basligi: {len(MADDE_RE.findall(m))} | kaynak: {os.path.basename(secilen[tur]['yol'])}")
ilk = next(iter(metinler))
print(f"\nOrnek ({ilk}, ilk 400 karakter):\n" + metinler[ilk][:400])

dengeleme: 413,884 karakter | MADDE basligi: 229 | kaynak: tr_dengeleme_uzlastirma.pdf
sebeke: 458,620 karakter | MADDE basligi: 126 | kaynak: tr_sebeke_yonetmeligi.pdf
veri_listesi: 25,357 karakter | MADDE basligi: 0 | kaynak: tr_seffaflik_veri_listesi.pdf
yan_hizmetler: 188,682 karakter | MADDE basligi: 90 | kaynak: _PortalAdmin_Uploads_Content_FastAccess_389cc81749363.docx

Ornek (dengeleme, ilk 400 karakter):
ELEKTRİK PİYASASI DENGELEME VE UZLAŞTIRMA YÖNETMELİĞİ
 
BİRİNCİ KISIM
Genel Hükümler
 
BİRİNCİ BÖLÜM
Amaç, Kapsam, Dayanak ve Tanımlar
Amaç
MADDE 1 –
 (1) Bu Yönetmeliğin amacı; aktif elektrik enerjisi arz ve talebinin dengelenmesine ve uzlaştırmanın gerçekleştirilmesine ilişkin usul
ve esaslarını belirlemektir.
Kapsam
MADDE 2 –
 (1) Bu Yönetmelik; dengeleme mekanizmasının ve uzlaştırmanın tarafla


## C. Madde Bazlı Parçalama

- **Temel birim madde.** Her parçanın başına belge adı ve madde etiketi ekleniyor, örneğin `[Elektrik Şebeke Yönetmeliği | MADDE 4]`. Bu başlık, parçanın hangi belgeye ait olduğunu embedding'e de taşıyor.
- **Uzun maddeler** (örneğin yüzlerce tanım içeren "Tanımlar" maddesi) 2.000 karakterlik, 200 karakter örtüşen alt parçalara bölünüyor. Madde numarası korunuyor.
- **Madde yapısı olmayan belgeler** (veri listesi gibi, en az 3 MADDE başlığı yoksa) 1.500 karakterlik, 200 karakter örtüşen bölümlere ayrılıyor.
- **Parça kimliği** metnin özetini (hash) içeriyor. Belge güncellenirse yalnızca değişen parçalar yeniden gömülüyor.

In [6]:
MAKS, ORTUSME = 2000, 200

def pencereler(metin, uzunluk, ortusme):
    if len(metin) <= uzunluk:
        return [metin]
    adim = uzunluk - ortusme
    return [metin[i:i + uzunluk] for i in range(0, len(metin) - ortusme, adim)]

def parcala(belge_turu, metin):
    ad = TURLER[belge_turu][0]
    kok = belge_turu
    eslesme = list(MADDE_RE.finditer(metin))
    birimler = []
    if len(eslesme) >= 3:
        giris = metin[:eslesme[0].start()].strip()
        if len(giris) > 200:
            birimler.append(("GIRIS", "", giris))
        for i, m in enumerate(eslesme):
            son = eslesme[i + 1].start() if i + 1 < len(eslesme) else len(metin)
            tur = re.sub(r"\s+", " ", m.group(1))
            birimler.append((tur, re.sub(r"\s+", "", m.group(2)), metin[m.start():son].strip()))
    else:
        for j, p in enumerate(pencereler(metin, 1500, ORTUSME), start=1):
            birimler.append(("BOLUM", str(j), p))
    parcalar = []
    for tur, no, govde in birimler:
        etiket = f"{tur} {no}".strip()
        alt = pencereler(govde, MAKS, ORTUSME) if tur != "BOLUM" else [govde]
        for k, a in enumerate(alt, start=1):
            baslik = f"[{ad} | {etiket}{f' (parca {k}/{len(alt)})' if len(alt) > 1 else ''}]"
            tam = baslik + "\n" + a
            h = hashlib.sha1(tam.encode("utf-8")).hexdigest()[:10]
            parcalar.append({"id": f"{kok}_{len(parcalar):04d}_{h}", "metin": tam,
                             "meta": {"dosya": kok, "kaynak": os.path.basename(secilen[kok]["yol"]), "belge": ad,
                                      "madde": etiket, "madde_turu": tur,
                                      "madde_no": no, "parca": k, "karakter": len(a)}})
    return parcalar

tum = []
for f, m in metinler.items():
    p = parcala(f, m)
    tum += p
    uz = [x["meta"]["karakter"] for x in p]
    print(f"{f}: {len(p)} parca | madde: {len({x['meta']['madde'] for x in p})} | "
          f"karakter ort {np.mean(uz):.0f}, maks {max(uz)}")
print("\nToplam parca:", len(tum))
if tum:
    print("\nOrnek parca:\n" + tum[min(5, len(tum) - 1)]["metin"][:500])

dengeleme: 348 parca | madde: 229 | karakter ort 1257, maks 2000
sebeke: 331 parca | madde: 126 | karakter ort 1508, maks 2000
veri_listesi: 20 parca | madde: 20 | karakter ort 1458, maks 1500
yan_hizmetler: 148 parca | madde: 91 | karakter ort 1351, maks 2000

Toplam parca: 847

Ornek parca:
[Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 4 (parca 3/18)]
gerçekleştirilebilen çıkış gücü değişimi ile elde edilen yedek kapasitenin alış-satışının gerçekleştiği ve sistem işletmecisi tarafından işletilen organize
toptan elektrik piyasasını,
t) 
(Değişik:RG-2/2/2020-31027) 
Dengeleme mekanizması: Vadeli elektrik piyasası ile ikili anlaşmaları tamamlayıcı nitelikteki, gün öncesi
piyasası, gün içi piyasası ve gerçek zamanlı dengelemeden oluşan faaliyetleri,
u) EIC: Oluşturulma


## D. Embedding ve ChromaDB

- **Model:** `gemini-embedding-001`, 768 boyut.
- **Normalizasyon:** Vektörler L2 ile normalize ediliyor. Birim vektörlerde ChromaDB'nin varsayılan L2 uzaklığı, kosinüs benzerliğiyle aynı sıralamayı veriyor. Benzerlik skoru = 1 − uzaklık / 2.
- **Kota:** 20'şer parçalık gruplar, gruplar arasında 1 saniye bekleme. Kota hatasında (429) 20, 60 ve 120 saniye beklenerek tekrar deneniyor.
- **Tekrar çalıştırma:** Koleksiyonda zaten bulunan parçalar yeniden gömülmüyor; artık belgede olmayan eski parçalar siliniyor.

In [7]:
gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def gom(metinler_listesi, gorev):
    for deneme, bekle in enumerate([0, 20, 60, 120], start=1):
        time.sleep(bekle)
        try:
            r = gemini.models.embed_content(
                model=EMBED_MODEL, contents=metinler_listesi,
                config=types.EmbedContentConfig(task_type=gorev, output_dimensionality=BOYUT))
            v = np.array([e.values for e in r.embeddings], dtype="float32")
            return v / np.linalg.norm(v, axis=1, keepdims=True)
        except Exception as e:
            mesaj = str(e)
            print(f"  deneme {deneme}: {mesaj[:120]}")
            if "429" not in mesaj and "RESOURCE_EXHAUSTED" not in mesaj:
                raise
    raise RuntimeError("Embedding istegi basarisiz")

chroma = chromadb.PersistentClient(path=os.path.join(RAG_DIR, "chroma_tr"))
kol = chroma.get_or_create_collection(name="tr_mevzuat")

guncel = {p["id"]: p for p in tum}
eski = set(kol.get(include=[])["ids"])
silinecek = sorted(eski - set(guncel))
eklenecek = [guncel[i] for i in guncel if i not in eski]
if silinecek:
    kol.delete(ids=silinecek)
print(f"Koleksiyonda: {len(eski)} | silinen eski parca: {len(silinecek)} | gomulecek yeni parca: {len(eklenecek)}")

t0 = time.time()
for i in range(0, len(eklenecek), 20):
    grup = eklenecek[i:i + 20]
    v = gom([p["metin"] for p in grup], "RETRIEVAL_DOCUMENT")
    kol.upsert(ids=[p["id"] for p in grup], embeddings=v.tolist(),
               documents=[p["metin"] for p in grup], metadatas=[p["meta"] for p in grup])
    time.sleep(1.0)
print(f"Tamamlandi: {time.time() - t0:.0f} sn | koleksiyondaki parca: {kol.count()}")

Koleksiyonda: 827 | silinen eski parca: 550 | gomulecek yeni parca: 570
Tamamlandi: 58 sn | koleksiyondaki parca: 847


## E. Getirme ve Duruma Duyarlı Sorgu

`build_rag_query_tr(durum)`, PJM projesindeki `build_rag_query`'nin Türkçe karşılığı. Agent'ın durumuna göre farklı mevzuat bölgelerini arıyor:

| Durum | Sorgunun odağı |
|---|---|
| `normal` | Dengeleme güç piyasası, yedek kapasite, yük alma/atma talimatları |
| `attention` | Yük atma talimatları, talep tarafı, sistem işletmecisinin dengeleme yükümlülükleri |
| `action_required` | Acil durum, acil durum talimatları, düşük frekans rölesi ile yük atma |
| `veri` | Verilerin yayımlanma zamanları (gerçekleşen tüketim, yük tahmin planı); Agent'ın veri uyarılarını açıklaması için |

In [8]:
@traceable(name="tr_mevzuat_retrieval")
def getir(sorgu, k=4, dosya=None):
    q = gom([sorgu], "RETRIEVAL_QUERY")[0]
    ek = {"where": {"dosya": dosya}} if dosya else {}
    r = kol.query(query_embeddings=[q.tolist()], n_results=k, **ek)
    return [{"id": i, "belge": m["belge"], "madde": m["madde"], "benzerlik": round(1 - d / 2, 4), "metin": t}
            for i, m, d, t in zip(r["ids"][0], r["metadatas"][0], r["distances"][0], r["documents"][0])]

def build_rag_query_tr(durum):
    sorgular = {
        "normal": "dengeleme güç piyasası yedek kapasite yük alma ve yük atma talimatları",
        "attention": "yük atma talimatı talep tarafı sistem işletmecisinin dengeleme yükümlülükleri",
        "action_required": "acil durum sistem kararlılığı ve emniyeti acil durum talimatı düşük frekans rölesi yük atma",
        "veri": "gerçekleşen tüketim ve gün öncesi yük tahmin planının yayımlanma zamanı",
    }
    return sorgular.get(durum, sorgular["normal"])

def baglam_metni(parcalar):
    return "\n---\n".join(f"[{p['belge']} | {p['madde']}] (benzerlik {p['benzerlik']})\n{p['metin']}" for p in parcalar)

for durum in ["normal", "attention", "action_required", "veri"]:
    s = build_rag_query_tr(durum)
    sonuc = getir(s, k=3)
    print(f"\n{durum}: '{s}'")
    for p in sonuc:
        print(f"   {p['benzerlik']:.3f}  {p['belge']} | {p['madde']}")


normal: 'dengeleme güç piyasası yedek kapasite yük alma ve yük atma talimatları'
   0.793  Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 72
   0.787  Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 73
   0.785  Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 71/A

attention: 'yük atma talimatı talep tarafı sistem işletmecisinin dengeleme yükümlülükleri'
   0.762  Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 74
   0.750  Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 4
   0.750  Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği | MADDE 73

action_required: 'acil durum sistem kararlılığı ve emniyeti acil durum talimatı düşük frekans rölesi yük atma'
   0.762  Elektrik Şebeke Yönetmeliği | MADDE 67
   0.754  Elektrik Şebeke Yönetmeliği | MADDE 62
   0.743  Elektrik Şebeke Yönetmeliği | MADDE 65

veri: 'gerçekleşen tüketim ve gün öncesi yük tahmin planının yayımlanma zamanı'
   0.758  Elektrik Piyasası Dengele

## F. Getirme Kalite Testi

Her test sorgusu için beklenen anahtar ifade, ilgili belgede **gerçekten geçtiği doğrulanmış** bir ifade. Bu ifadeler araştırma sırasında resmî metinlerden alındı:
- Dengeleme güç piyasasının tanımı "15 dakika" içeriyor.
- Primer frekans kontrol rezervi "±200 mHz" ile tanımlanıyor.
- Şebeke Yönetmeliği'nde "49,8 Hz" eşiği geçiyor.
- Veri listesinde gerçekleşen tüketim "S+2", yük tahmin planı "G-1" olarak geçiyor.

Ölçüt: Anahtar ifade ilk k parçanın (k = 4) birinde geçiyor mu (hit@4). Kaçırılan her sorgu için neden de yazdırılıyor: ifade herhangi bir parçada varsa sorun **getirmede**, hiçbir parçada yoksa sorun **PDF okuma veya parçalamada**. Karşılaştırmadan önce Türkçe küçük harfe çevriliyor ve boşluklar tekleştiriliyor; PDF'lerde ifadeler satır sonuyla bölünebiliyor. Beklenen oran yüksek olmalı. Kaçırılan sorgular, parçalama veya sorgu ifadesinin iyileştirilmesi için ipucu verir.

In [9]:
TESTLER = [
    ("Acil durum tanımı: sistem kararlılığı ve emniyeti açısından tehlike arz eden durumlar", "acil durum"),
    ("Dengeleme güç piyasasında yedek kapasite kaç dakika içinde devreye alınır", "15 dakika"),
    ("Primer frekans kontrol rezerv kapasitesi hangi frekans sapmasına göre tanımlanır", "200 mhz"),
    ("Sistem frekansı 49,8 Hz'nin altına düştüğünde üretim tesisleri ne yapar", "49,8"),
    ("Gerçekleşen tüketim verisi ne kadar gecikmeyle yayımlanır", "s+2"),
    ("Gün öncesi yük tahmin planı ne zaman yayımlanır", "g-1"),
    ("Düşük frekans rölesi ile tüketim yüklerinin kesilmesi", "düşük frekans rölesi"),
    ("Yük atma talimatı nedir", "yük atma"),
]
satirlar = []
for sorgu, anahtar in TESTLER:
    sonuc = getir(sorgu, k=4)
    sira = next((i + 1 for i, p in enumerate(sonuc) if anahtar in tr_kucuk(p["metin"])), None)
    parcada_var = any(anahtar in tr_kucuk(p["metin"]) for p in tum)
    satirlar.append({"sorgu": sorgu[:60], "anahtar": anahtar, "bulundu": sira is not None, "sira": sira,
                     "herhangi_bir_parcada_var": parcada_var,
                     "ilk_sonuc": f"{sonuc[0]['belge'][:28]} | {sonuc[0]['madde']}" if sonuc else "",
                     "benzerlik": sonuc[0]["benzerlik"] if sonuc else None})
kalite = pd.DataFrame(satirlar)
print(kalite.to_string(index=False))
print(f"\nhit@4: {kalite['bulundu'].mean() * 100:.0f}% ({int(kalite['bulundu'].sum())}/{len(kalite)})")
kacan = kalite[~kalite["bulundu"]]
if len(kacan):
    print("Kacirilan sorgularin nedeni:")
    for _, r in kacan.iterrows():
        neden = ("GETIRME: ifade bir parcada var ama ilk 4'e girmedi" if r["herhangi_bir_parcada_var"]
                 else "PARCALAMA/PDF: ifade hicbir parcada yok (PDF metni veya bolunme kontrol edilmeli)")
        print(f"  '{r['anahtar']}': {neden}")
kalite.to_csv(os.path.join(KOK, "metrics", "tr_rag_kalite_testi.csv"), index=False)

                                                       sorgu              anahtar  bulundu  sira  herhangi_bir_parcada_var                                 ilk_sonuc  benzerlik
Acil durum tanımı: sistem kararlılığı ve emniyeti açısından            acil durum     True     1                      True   Elektrik Piyasası Dengeleme  | MADDE 75     0.7003
Dengeleme güç piyasasında yedek kapasite kaç dakika içinde d            15 dakika     True     1                      True   Elektrik Piyasası Dengeleme  | MADDE 70     0.7475
Primer frekans kontrol rezerv kapasitesi hangi frekans sapma              200 mhz     True     1                      True   Elektrik Şebeke Yönetmeliği | MADDE 105     0.7794
Sistem frekansı 49,8 Hz'nin altına düştüğünde üretim tesisle                 49,8     True     2                      True    Elektrik Şebeke Yönetmeliği | MADDE 20     0.7660
   Gerçekleşen tüketim verisi ne kadar gecikmeyle yayımlanır                  s+2     True     1                      Tr

## G. Agent İçin Paylaşılan Modül

Agent (`6_agent_tr.ipynb`) ve FastAPI servisi ayrı süreçlerde çalışacak. Bu yüzden getirme fonksiyonları diskteki ChromaDB'ye bağlanan ayrı bir dosyaya yazılıyor: `notepads/tr_rag_retrieval.py`. PJM projesindeki `rag_retrieval.py` yapısının Türkçe karşılığı. Son hücre, modülü yeni bir import ile test ediyor.

In [10]:
MODUL = r"""# tr_rag_retrieval.py - 5_rag_tr.ipynb tarafindan uretildi. Elle degistirmeyin; notebook'u yeniden calistirin.
import os
import numpy as np
import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langsmith import traceable

KOK = r"__KOK__"
load_dotenv(os.path.join(KOK, "notepads", ".env"))
os.environ.setdefault("LANGSMITH_PROJECT", "tr-grid-agent")
EMBED_MODEL, BOYUT = "__MODEL__", __BOYUT__

_gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
_kol = chromadb.PersistentClient(path=os.path.join(KOK, "rag_docs", "chroma_tr")).get_or_create_collection(name="tr_mevzuat")

def _gom_sorgu(metin):
    r = _gemini.models.embed_content(model=EMBED_MODEL, contents=[metin],
                                     config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=BOYUT))
    v = np.array(r.embeddings[0].values, dtype="float32")
    return v / np.linalg.norm(v)

@traceable(name="tr_mevzuat_retrieval")
def getir(sorgu, k=4, dosya=None):
    ek = {"where": {"dosya": dosya}} if dosya else {}
    r = _kol.query(query_embeddings=[_gom_sorgu(sorgu).tolist()], n_results=k, **ek)
    return [{"id": i, "belge": m["belge"], "madde": m["madde"], "benzerlik": round(1 - d / 2, 4), "metin": t}
            for i, m, d, t in zip(r["ids"][0], r["metadatas"][0], r["distances"][0], r["documents"][0])]

def build_rag_query_tr(durum):
    sorgular = __SORGULAR__
    return sorgular.get(durum, sorgular["normal"])

def baglam_metni(parcalar):
    return "\n---\n".join(f"[{p['belge']} | {p['madde']}] (benzerlik {p['benzerlik']})\n{p['metin']}" for p in parcalar)
"""
if kol.count() == 0:
    raise RuntimeError("Koleksiyon bos; modul yazilmadi. Once belgeleri indirip D bolumunu calistirin.")
sorgular = {d: build_rag_query_tr(d) for d in ["normal", "attention", "action_required", "veri"]}
icerik = (MODUL.replace("__KOK__", KOK).replace("__MODEL__", EMBED_MODEL).replace("__BOYUT__", str(BOYUT))
               .replace("__SORGULAR__", json.dumps(sorgular, ensure_ascii=False)))
yol = os.path.join(KOK, "notepads", "tr_rag_retrieval.py")
with open(yol, "w", encoding="utf-8") as fh:
    fh.write(icerik)
print("Yazildi:", yol)

sys.path.insert(0, os.path.join(KOK, "notepads"))
if "tr_rag_retrieval" in sys.modules:
    del sys.modules["tr_rag_retrieval"]
import tr_rag_retrieval as trr
test = trr.getir(trr.build_rag_query_tr("action_required"), k=2)
print("Modul testi:", [(p["belge"][:30], p["madde"], p["benzerlik"]) for p in test])

Yazildi: C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus\notepads\tr_rag_retrieval.py
Modul testi: [('Elektrik Şebeke Yönetmeliği', 'MADDE 67', 0.7617), ('Elektrik Şebeke Yönetmeliği', 'MADDE 62', 0.7536)]
